# NDgpu — GPU mesh solver & a *fair* CPU-vs-GPU benchmark (Google Colab)

This session made the **unstructured mesh solver GPU-native**: the general-geometry
finite-volume solver (`UnstructuredDiffusionSolver`) used to factorize a SciPy
sparse matrix on the CPU (`splu`); it now applies its within-group operator
**matrix-free** through the same NumPy/CuPy code path as the structured lattices.
The irregular face coupling is stored as a row-wise **ELLPACK adjacency** and the
apply is a pure **gather** — each cell reads its neighbours' flux, no scatter, so
no GPU atomics and a coalesced write (the access pattern that makes the structured
stencils fast on CUDA). So the whole local-refinement / arbitrary-mesh track
(every HP-MR drum study run on a mesh) now runs on the GPU.

This notebook does two things:

1. **Validates** the GPU mesh solver (same `k` as CPU, to 1e-9), including its new
   **3D** support (tetrahedra / hexahedra / prisms) checked against the structured
   solver on the HP-MR core, and exercises the other new-this-session GPU
   capabilities — **TriSP3** transport and the **SPH** self-shielding correction.
2. Runs a CPU-vs-GPU **performance study that takes its own fairness seriously**:
   what "one CPU" even means here, how many threads the CPU baseline actually
   uses, and why the *bandwidth ratio* — not the raw speed-up — is the honest
   hardware number.

**To run:** Runtime → Change runtime type → **T4 GPU**, then Run all. Upload
`dist/ndgpu-src.zip` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()          # upload dist/ndgpu-src.zip
zip_name = next(iter(uploaded))
%pip install -q {zip_name}
try:
    import cupy
except ImportError:
    %pip install -q cupy-cuda12x
try:
    import threadpoolctl
except ImportError:
    %pip install -q threadpoolctl

import os
print("\n===== GPU =====")
!nvidia-smi --query-gpu=name,memory.total,power.max_limit --format=csv,noheader
print("\n===== CPU =====")
!lscpu | grep -E "^Model name|^Socket|^Core|^Thread|^CPU\(s\)|MHz" | sed 's/  */ /g'
print("logical CPUs visible to Python (os.cpu_count):", os.cpu_count())

## 1. Correctness first — the GPU mesh solver gives the same answer

Two checks on `UnstructuredDiffusionSolver`, run on **both** backends:

* a bare homogeneous medium with **reflective** boundaries must return exactly
  `k_inf`, on any geometry;
* the **HP-MR locally-refined drum mesh** (coarse triangles everywhere, the 1 cm
  B4C absorber band split to 2:1 hanging nodes) must give a `k` that is identical
  on CPU and GPU — the one-code-path guarantee. If this holds, the speed-up cells
  below are a pure hardware measurement, not an algorithm change.

In [ ]:
import numpy as np
from ndgpu import k_infinite, ONE_GROUP_DEMO
from ndgpu.mesh import UnstructuredDiffusionSolver, assemble_mesh
from ndgpu.benchmarks.hpmr import hpmr_locally_refined_mesh


def quad_mesh(n, L):
    # an n x n Cartesian mesh of quads, assembled as an unstructured mesh
    dx = L / n
    nid, coords = {}, []
    def gid(i, j):
        if (i, j) not in nid:
            nid[(i, j)] = len(coords); coords.append((i * dx, j * dx))
        return nid[(i, j)]
    cells = [(gid(i, j), gid(i + 1, j), gid(i + 1, j + 1), gid(i, j + 1))
             for i in range(n) for j in range(n)]
    return assemble_mesh(coords, cells, [0] * len(cells))


# (a) reflective bare medium -> k_inf on both devices
m = quad_mesh(24, 50.0)
kinf = k_infinite(ONE_GROUP_DEMO)
for dev in ("cpu", "gpu"):
    k = UnstructuredDiffusionSolver(m, [ONE_GROUP_DEMO], np.zeros(m.n_cells, int),
                                    alpha_boundary=0.0, device=dev).solve(tol_k=1e-10).k_eff
    print(f"reflective quad mesh [{dev}]: k = {k:.8f}   (k_inf = {kinf:.8f}, {abs(k-kinf)*1e5:.2e} pcm)")

# (b) HP-MR locally-refined drum mesh: CPU and GPU must agree
mesh, cm, mats, alpha = hpmr_locally_refined_mesh(refine=4, drum_angle_deg=180.0,
                                                  refine_drums=True)
res = {}
for dev in ("cpu", "gpu"):
    r = UnstructuredDiffusionSolver(mesh, mats, cm, alpha, device=dev).solve(tol_k=1e-9)
    res[dev] = r
    print(f"HP-MR refined mesh ({mesh.n_cells:,} cells) [{dev}]: "
          f"k = {r.k_eff:.8f}   {r.solve_seconds:.2f}s   [{r.device}]")

assert abs(res["cpu"].k_eff - res["gpu"].k_eff) < 1e-9
print("\nCPU and GPU agree to < 1e-9 in k_eff (< 1e-4 pcm) — one code path, same answer.")

## 2. Is it fair to compare *one* GPU to *one* CPU?

There is no single "fair" — it depends on the question. Three defensible framings:

**(1) "Same code, one flag" — the *usage* comparison.** Flip `device='cpu'→'gpu'`
and the byte-for-byte identical iteration sequence runs on the other backend
(CuPy mirrors the NumPy API). This answers *"what do I gain by moving my existing
NDgpu run onto the GPU I have?"* — honest about the software, silent about
hardware parity: the two chips are not matched on cores, die area, power, or price.

**(2) Chip vs chip.** You must say what "one CPU" is — see the `lscpu` output
above. A Colab box is typically **~2 vCPUs** of a shared Intel Xeon (a few tens of
GB/s of DDR4), against a full **NVIDIA T4** (2560 CUDA cores, 16 GB, **~320 GB/s**,
70 W). Even "chip vs chip" is lopsided: a T4 is a far bigger, newer die than 2
vCPUs, so a raw ratio flatters the GPU.

**(3) Per-resource — bandwidth / watt / dollar.** For *this* solver the binding
resource is **memory bandwidth**, so the bandwidth-normalized number is the
physically meaningful one (Section 4).

### How many CPU threads are we actually using? (≈ one)

The matrix-free hot loop is elementwise arithmetic (`diag*phi`), a neighbour
**gather** (`phi[nbr]`) with a reduction, and dot-product / `sum` reductions.
**Stock NumPy runs all of these on a single core** — they are serial C loops;
only BLAS-3 calls (matmul) use the
MKL/OpenBLAS thread pool, and this solver makes none. So our **CPU baseline is
effectively single-threaded**, regardless of how many cores the box has. The next
cell proves it: capping the thread pool to 1 vs all cores leaves the solve time
unchanged.

The consequence is stated plainly: a hand-parallelized CPU code (Numba / OpenMP /
numexpr over the same stencil) could reclaim roughly the socket's core count, so
**the raw speed-up over-states the chip-vs-chip advantage by up to that factor.**
We show the single-core NumPy baseline rather than hide it.

In [ ]:
import os, numpy as np
from threadpoolctl import threadpool_info, threadpool_limits

print("native thread pools NumPy could use:",
      [f"{d.get('internal_api')}({d.get('num_threads')})" for d in threadpool_info()])

# Same CPU solve under a 1-thread vs all-cores pool. Elementwise + bincount +
# reductions don't use these pools, so we expect ~no change -> the workload is
# single-core-bound, which is what makes "1 GPU vs ~1 CPU core" the true picture.
big, cmb, matsb, ab = hpmr_locally_refined_mesh(refine=6, drum_angle_deg=180.0,
                                                refine_drums=True)
def cpu_solve_time():
    s = UnstructuredDiffusionSolver(big, matsb, cmb, ab, device="cpu")
    s.solve(max_outer=3, tol_k=0.0)                    # warm-up (untimed)
    return s.solve(tol_k=1e-8).solve_seconds

print(f"\nmesh: {big.n_cells:,} cells")
for lim in (1, os.cpu_count()):
    with threadpool_limits(limits=lim):
        t = min(cpu_solve_time() for _ in range(2))
    print(f"  CPU solve, thread pool capped to {lim:>2}: {t:.2f} s")
print("\n-> time is flat in thread count: the CPU leg is one core's worth of work.")

## 3. The solver-level speed-up (with all the caveats above in force)

Bare rectangular 2-group reactor (vacuum boundary — a real flux shape, so the CG
does real work) on quad meshes of growing size, `device='cpu'` vs `'gpu'`. Each
timed solve follows an untimed **warm-up** (compiles CUDA kernels, allocates
buffers) and `solve_seconds` spans a device `synchronize()`, so it is honest
wall-clock time. `k(cpu)==k(gpu)` and the outer-iteration count is identical on
both, so the ratio is pure hardware.

The sweep runs up to ~200k unknowns. **The GPU advantage grows with size**: small
meshes are dominated by Python/kernel-launch overhead (the GPU can even lose),
and only once the mesh is large enough to keep the cores fed does the gather
apply reach the bandwidth-bound regime where it wins. Read this as the
**framing-(1)** number ("same code, one flag"), remembering from Section 2 that
the CPU side is ~one core. (The largest row is the slow one — ~20 s on the
single-core CPU leg.)

In [ ]:
from ndgpu import PWR_TWO_GROUP

hdr = (f"{'mesh':>10}{'unknowns':>11}{'cpu [s]':>9}{'gpu [s]':>9}"
       f"{'speed-up':>10}{'outer':>7}{'dk(cpu-gpu)':>13}")
print(hdr); print("-" * len(hdr))
for n in (48, 96, 160, 224, 320):
    m = quad_mesh(n, 120.0)                            # assembly is Python, untimed
    cm = np.zeros(m.n_cells, int)
    t, kk, outit = {}, {}, {}
    for dev in ("cpu", "gpu"):
        s = UnstructuredDiffusionSolver(m, [PWR_TWO_GROUP], cm,
                                        alpha_boundary=0.5, device=dev)
        s.solve(max_outer=3, tol_k=0.0)                # warm-up
        r = s.solve(tol_k=1e-7, tol_source=1e-6)
        t[dev], kk[dev], outit[dev] = r.solve_seconds, r.k_eff, r.outer_iterations
    print(f"{n:>4}x{n:<4}{2*m.n_cells:>11,}{t['cpu']:>9.2f}{t['gpu']:>9.2f}"
          f"{t['cpu']/t['gpu']:>9.1f}x{outit['cpu']:>7}{abs(kk['cpu']-kk['gpu']):>13.1e}")

## 4. The *physically fair* number: achieved memory bandwidth

The operator apply has **low arithmetic intensity** (a few flops per byte moved),
so it is **memory-bandwidth bound** on both devices. For such a kernel the
achievable speed-up ceiling is the **bandwidth ratio**, not the FLOP ratio or the
core count:

$$\text{speed-up}_\text{ceiling} \;\approx\; \frac{BW_\text{GPU}}{BW_\text{CPU}}
   \;\approx\; \frac{320}{\sim 20\text{–}40}\;\approx\; 8\text{–}16\times.$$

Below we time **only the within-group apply** (the hot kernel) on each device and
report its *achieved* GB/s from a stated per-cell byte model. Comparing those two
GB/s figures is the apples-to-apples, size-independent hardware statement. A
solver-level speed-up (Section 3) that lands **near** the bandwidth ratio is
"as fast as the hardware allows"; one far **above** it would be a red flag —
usually an under-fed (e.g. accidentally single-threaded on a many-core box) CPU
baseline. Here the CPU baseline genuinely *is* one core, which we have owned up to.

In [ ]:
import time
from ndgpu.backend import get_backend, synchronize

def apply_bandwidth(dev, n=200, reps=100):
    xp = get_backend(dev)
    m = quad_mesh(n, 120.0)
    s = UnstructuredDiffusionSolver(m, [PWR_TWO_GROUP], np.zeros(m.n_cells, int),
                                    alpha_boundary=0.5, device=dev)
    op = s.ops[0]
    x = xp.asarray(np.random.rand(m.n_cells))
    op.apply(x); synchronize(xp)                       # warm-up
    t0 = time.perf_counter()
    for _ in range(reps):
        y = op.apply(x)
    synchronize(xp)
    dt = (time.perf_counter() - t0) / reps
    # bytes / apply: the ELLPACK gather reads the neighbour-index and weight
    # tables (op.nbr, op.w_ell) and the diagonal once, plus phi (diag term +
    # gathered) and the output write. Use the operators' actual array sizes.
    b = op.nbr.nbytes + op.w_ell.nbytes + op.diag.nbytes + 3 * x.nbytes
    return dt, m.n_cells, b / dt / 1e9

print(f"{'device':>8}{'cells':>10}{'apply [ms]':>12}{'~GB/s':>9}")
gbs = {}
for dev in ("cpu", "gpu"):
    dt, nc, g = apply_bandwidth(dev)
    gbs[dev] = g
    print(f"{dev:>8}{nc:>10,}{dt*1e3:>12.3f}{g:>9.1f}")
print(f"\napply bandwidth ratio  GPU/CPU = {gbs['gpu']/gbs['cpu']:.1f}x"
      f"  (this is the hardware-fair number)")

## 5. Where the GPU gain is largest: the heavy 3D HP-MR core

The 2D unstructured mesh solver above is one geometry track; the biggest
performance gain lives in the **large, bandwidth-bound 3D** problems. The full
HP-MR microreactor on the structured triangular-prism solver
(`TriDiffusionEigenSolver`) is exactly that — hundreds of thousands of unknowns,
a coalesced stencil apply, many power-iteration steps. This is where a GPU
amortizes its launch overhead completely and runs near its memory-bandwidth roof.

*(3D uses the structured tri-prism solver — the unstructured `Mesh` is 2D. Same
one-code-path story, `device='cpu'|'gpu'`.)*

We also add a `dtype=float32` GPU column: it roughly doubles throughput (half the
bytes moved on a bandwidth-bound kernel), at a real accuracy cost — `k` shifts by
tens of pcm because the solve converges to the float32 noise floor. Reported, not
hidden: it is a throughput/accuracy trade, useful for scoping runs, not for a
final eigenvalue.

In [ ]:
from ndgpu.tri import TriDiffusionEigenSolver
from ndgpu.benchmarks import build_hpmr3d

TOL3D = dict(tol_k=1e-6, tol_source=1e-5)
hdr = (f"{'HP-MR 3D':>11}{'unknowns':>11}{'cpu [s]':>9}{'gpu64 [s]':>11}"
       f"{'gpu32 [s]':>11}{'sp(64)':>9}{'sp(32)':>9}{'d k32 [pcm]':>12}")
print(hdr); print("-" * len(hdr))
for refine, nz in [(4, 10), (4, 20), (5, 20)]:
    p = build_hpmr3d(refine=refine, nz=nz)            # raster absorber (refine>=4)
    unk = int(p.active.sum()) * 2
    t, kk = {}, {}
    for dev, dt in [("cpu", np.float64), ("gpu", np.float64), ("gpu", np.float32)]:
        s = TriDiffusionEigenSolver(p.grid, p.materials, p.material_map,
                                    active=p.active, mask_bc=p.mask_bc, bc=p.bc,
                                    device=dev, dtype=dt)
        s.solve(max_outer=3, tol_k=0.0)               # warm-up (untimed)
        r = s.solve(**TOL3D)
        key = dev + ("32" if dt is np.float32 else "")
        t[key], kk[key] = r.solve_seconds, r.k_eff
    dk32 = (kk["gpu32"] - kk["gpu"]) * 1e5
    print(f"r={refine} nz={nz:<3}{unk:>13,}{t['cpu']:>9.2f}{t['gpu']:>11.2f}"
          f"{t['gpu32']:>11.2f}{t['cpu']/t['gpu']:>8.1f}x{t['cpu']/t['gpu32']:>8.1f}x"
          f"{dk32:>+12.0f}")
    assert abs(kk["cpu"] - kk["gpu"]) < 1e-9          # float64 CPU==GPU

## 6. Structured vs unstructured on the *same* 3D core

The unstructured mesh solver is now **3D** (tetrahedra / hexahedra / prisms), so
the HP-MR core can be solved two ways: the **structured** triangular-prism solver,
and the **unstructured** mesh solver on the identical geometry extruded into
wedges. They share the two-point-flux discretization, so they return the same `k`
(asserted). The question this cell answers: now that the mesh apply is a *gather*
(Section on the ELLPACK reformulation), does the general-geometry solver earn a
GPU speed-up in the same ballpark as the hand-written structured stencil? If so,
an arbitrary mesh no longer costs you the GPU.

*(Building the wedge mesh is one-time Python setup — tens of seconds, untimed.)*

In [ ]:
from ndgpu.mesh import assemble_mesh_3d, UnstructuredDiffusionSolver
from ndgpu.tri import TriDiffusionEigenSolver
from ndgpu.benchmarks.hpmr import (hpmr_locally_refined_mesh, build_hpmr3d,
    _placeholder_materials, FUEL, CENTRAL, AXIAL_REFLECTOR, TOTAL_HEIGHT,
    AXIAL_REFLECTOR_HEIGHT)

refine, nz, angle = 4, 10, 180.0
TOL3D = dict(tol_k=1e-6, tol_source=1e-5)

def extrude_wedges(coords2d, tris, cmat2d, nz, height):
    coords2d = np.asarray(coords2d, float); n2 = len(coords2d); dz = height / nz
    c3 = np.zeros(((nz + 1) * n2, 3))
    for lev in range(nz + 1):
        c3[lev*n2:(lev+1)*n2, :2] = coords2d; c3[lev*n2:(lev+1)*n2, 2] = lev*dz
    zc = (np.arange(nz) + 0.5) * dz
    refl = (zc < AXIAL_REFLECTOR_HEIGHT) | (zc > height - AXIAL_REFLECTOR_HEIGHT)
    cells, cmat = [], []
    for t, (a, b, c) in enumerate(tris):
        for lev in range(nz):
            o0, o1 = lev*n2, (lev+1)*n2
            cells.append((a+o0, b+o0, c+o0, a+o1, b+o1, c+o1))
            m = cmat2d[t]
            cmat.append(AXIAL_REFLECTOR if (m in (FUEL, CENTRAL) and refl[lev]) else m)
    return c3, cells, np.array(cmat)

# two models of the identical HP-MR core (drums inserted)
mesh2d, cmat2d, _, _ = hpmr_locally_refined_mesh(refine=refine, drum_angle_deg=angle,
                                                 refine_drums=False)
mats3d = _placeholder_materials(three_d=True)
c3, cells3, cmat3 = extrude_wedges(mesh2d.coords, mesh2d.cells, cmat2d, nz, TOTAL_HEIGHT)
mesh3d = assemble_mesh_3d(c3, cells3, cmat3)             # ~tens of seconds, untimed
p = build_hpmr3d(refine=refine, drum_angle_deg=angle, nz=nz)

print(f"HP-MR 3D core: {mesh3d.n_cells:,} cells, drums inserted\n")
print(f"{'solver':>22}{'k_eff':>11}{'cpu [s]':>9}{'gpu [s]':>9}{'speed-up':>10}")
print("-" * 62)
kk = {}
builders = [
    ("structured tri-prism", lambda dev: TriDiffusionEigenSolver(
        p.grid, p.materials, p.material_map, active=p.active, mask_bc=p.mask_bc,
        bc=p.bc, device=dev)),
    ("unstructured wedges", lambda dev: UnstructuredDiffusionSolver(
        mesh3d, mats3d, cmat3, alpha_boundary=0.5, device=dev)),
]
for name, make in builders:
    t = {}
    for dev in ("cpu", "gpu"):
        s = make(dev); s.solve(max_outer=3, tol_k=0.0)  # warm-up (untimed)
        r = s.solve(**TOL3D); t[dev] = r.solve_seconds; kk[name] = r.k_eff
    print(f"{name:>22}{kk[name]:>11.6f}{t['cpu']:>9.2f}{t['gpu']:>9.2f}{t['cpu']/t['gpu']:>9.1f}x")
dk = (kk["structured tri-prism"] - kk["unstructured wedges"]) * 1e5
print(f"\nsame core, two discretizations: d k = {dk:+.1f} pcm (must be ~0)")

## 7. The other new GPU capabilities: TriSP3 transport + SPH self-shielding

Both landed this session and both run on the GPU. On the 2D HP-MR (drums inserted):
**TriSP3** is the transport reference; plain **TriDiffusion** misses the near-black
B4C arc's self-shielding by ~120 pcm of drum worth; the **SPH** correction folds
that angular difference into the diffusion cross sections so coarse diffusion
reproduces the SP3 eigenvalue to a few pcm — all solved on the GPU.

In [ ]:
from ndgpu import (TriSP3EigenSolver, TriDiffusionEigenSolver,
                   flux_weighted_homogenize, region_average, sph_correct)
from ndgpu.benchmarks.hpmr import build_hpmr2d

p = build_hpmr2d(refine=4, drum_angle_deg=180.0, absorber="raster")
common = dict(active=p.active, mask_bc=p.mask_bc, device="gpu")
TOL = dict(tol_k=1e-9, tol_source=1e-8)

ref = TriSP3EigenSolver(p.grid, p.materials, p.material_map, **common).solve(**TOL)
dif = TriDiffusionEigenSolver(p.grid, p.materials, p.material_map, **common).solve(**TOL)
print(f"GPU TriSP3   k = {ref.k_eff:.6f}   {ref.solve_seconds:.2f}s   [{ref.device}]")
print(f"GPU TriDiff  k = {dif.k_eff:.6f}   (diffusion - SP3 = {(dif.k_eff-ref.k_eff)*1e5:+.0f} pcm)")

region = p.material_map                                # one region per material type
hmats, rflux, _ = flux_weighted_homogenize(ref.flux_numpy, p.materials, p.material_map,
                                           region, cell_volume=p.grid.cell_volume)
def coarse_solve(materials):
    r = TriDiffusionEigenSolver(p.grid, materials, region, **common).solve(**TOL)
    return region_average(r.flux_numpy, region), r.k_eff

out = sph_correct(hmats, region, rflux, coarse_solve, tol=1e-7, depth=6)
print(f"GPU SPH-corr k = {out.k_eff:.6f}   (residual vs SP3 = {(out.k_eff-ref.k_eff)*1e5:+.1f} pcm,"
      f" {out.iterations} SPH iters, converged={out.converged})")

## Verdict — what these numbers do and don't say

* **The one-code-path claim is real.** Every GPU `k` above equals its CPU twin to
  < 1e-9 in `k_eff` (< 1e-4 pcm); the CPU test suite exercises byte-for-byte the
  kernels that run on the GPU. The speed-up is a hardware measurement, not a
  different algorithm.
* **"1 GPU vs 1 CPU" is only meaningful once you name the CPU.** Ours is
  effectively **one core** (Section 2's thread test), because the workload is
  memory-bound elementwise/gather/reduction work that stock NumPy does not
  thread. The raw Section-3 speed-up is the honest *"flip one flag"* figure, but
  it over-states the chip-vs-chip advantage by roughly the core count a threaded
  CPU code could have used.
* **The bandwidth ratio (Section 4) is the hardware-fair number.** This solver is
  bandwidth-bound, so its ceiling is `BW_GPU / BW_CPU` (~8–16× for a T4 over a
  couple of DDR4 channels), and the measured apply ratio should sit near it. A
  solver speed-up much larger than the bandwidth ratio would signal an unfair
  (under-fed) CPU baseline, not GPU magic.
* **Bigger problems, bigger wins.** The advantage grows with size (Sections 3 &
  5): small/coarse meshes are dominated by Python and kernel-launch overhead and
  can be CPU-faster, while the heavy 3D HP-MR core (hundreds of thousands of
  unknowns) is where the GPU runs near its bandwidth roof and the speed-up is
  largest. Scope GPU claims to that regime.
* **The mesh solver needed the right kernel, not just the right device.** Moving
  it to CuPy was not enough: the first cut scattered face contributions with
  `bincount` (GPU atomics), which throttled it. Reformulating the apply as a
  **gather over an ELLPACK adjacency** — the structured stencils' access pattern —
  is what lets the general-geometry track actually use the GPU. A reminder that
  "GPU-native" is about the memory-access pattern, not the array library.